In [75]:
import numpy as np
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, adfuller
import matplotlib.pyplot as plt

In [76]:
sector_map = {

    "BANKING": [
        "HDFCBANK.NS",
        "ICICIBANK.NS",
        "SBIN.NS",
        "AXISBANK.NS",
        "KOTAKBANK.NS",
        "INDUSINDBK.NS",
        "BANKBARODA.NS",
        "FEDERALBNK.NS"
    ],

    "IT": [
        "TCS.NS",
        "INFY.NS",
        "HCLTECH.NS",
        "WIPRO.NS",
        "TECHM.NS",
        "LTIM.NS",
        "COFORGE.NS",
        "PERSISTENT.NS"
    ],

    "AUTO": [
        "MARUTI.NS",
        "M&M.NS",
        "BAJAJ-AUTO.NS",
        "EICHERMOT.NS",
        "HEROMOTOCO.NS",
        "TVSMOTOR.NS",
        "ASHOKLEY.NS",
        "MOTHERSON.NS"
    ],

    "REALTY": [
        "DLF.NS",
        "LODHA.NS",
        "OBEROIRLTY.NS",
        "GODREJPROP.NS",
        "PRESTIGE.NS",
        "BRIGADE.NS",
        "SOBHA.NS",
        "PHOENIXLTD.NS"
    ],

    "FMCG": [
        "HINDUNILVR.NS",
        "ITC.NS",
        "NESTLEIND.NS",
        "BRITANNIA.NS",
        "DABUR.NS",
        "MARICO.NS",
        "GODREJCP.NS",
        "TATACONSUM.NS"
    ],

    "ENERGY_UTILS": [
        "RELIANCE.NS",
        "ONGC.NS",
        "IOC.NS",
        "BPCL.NS",
        "GAIL.NS",
        "NTPC.NS",
        "POWERGRID.NS",
        "COALINDIA.NS"
    ],

    "PHARMA": [
        "SUNPHARMA.NS",
        "DRREDDY.NS",
        "CIPLA.NS",
        "DIVISLAB.NS",
        "APOLLOHOSP.NS",
        "LUPIN.NS",
        "ALKEM.NS",
        "AUROPHARMA.NS"
    ]
}

all_tickers = sorted({t for lst in sector_map.values() for t in lst})
print("Total tickers:", len(all_tickers))

Total tickers: 56


In [77]:
prices = yf.download(
    all_tickers,
    start="2022-01-01",
    end="2025-12-31",
    auto_adjust=True,
    progress=False
)["Close"]

if isinstance(prices, pd.Series):
    prices = prices.to_frame()

prices = prices.sort_index().dropna(how="all")


In [78]:
train = prices.loc["2022-01-01":"2024-12-31"].copy()
test  = prices.loc["2025-01-01":"2025-12-31"].copy()

print("Train:", train.index.min().date(), "->", train.index.max().date(), "shape", train.shape)
print("Test :", test.index.min().date(),  "->", test.index.max().date(),  "shape", test.shape)

Train: 2022-01-03 -> 2024-12-31 shape (739, 56)
Test : 2025-01-01 -> 2025-12-30 shape (248, 56)


In [ ]:
from itertools import combinations

ADF_P_TH = 0.05
MIN_TRAIN_DAYS = 252

rows = []

for sector, tickers in sector_map.items():
    
    tickers = [t for t in tickers if t in train.columns]
    
    for x, y in combinations(tickers, 2):
        
        df = train[[x, y]].dropna()
        if len(df) < MIN_TRAIN_DAYS:
            continue
        
        # OLS regression
        Y = df[y].values
        X = sm.add_constant(df[x].values)
        model = sm.OLS(Y, X).fit()
        alpha, beta = model.params
        
        # Residual (spread)
        resid = Y - (alpha + beta * df[x].values)
        
        # ADF on residual
        adf_stat, pval, *_ = adfuller(resid, regression="c", autolag="AIC")
        
        if pval < ADF_P_TH:
            rows.append({
                "sector": sector,
                "x": x,
                "y": y,
                "alpha": float(alpha),
                "beta": float(beta),
                "adf_p": float(pval)
            })

selected_df = pd.DataFrame(rows).sort_values(["sector", "adf_p"]).reset_index(drop=True)

selected_df = (
    pd.DataFrame(rows)
    .sort_values(["sector", "adf_p"])
    .groupby("sector")
    .head(5)
    .reset_index(drop=True)
)
selected_df

,sector,x,y,alpha,beta,adf_p
0,AUTO,M&M.NS,ASHOKLEY.NS,36.915711,0.026650,0.008386
1,AUTO,ASHOKLEY.NS,MOTHERSON.NS,-29.098408,1.259047,0.010417
2,AUTO,M&M.NS,EICHERMOT.NS,1817.517834,1.056151,0.013623
3,AUTO,EICHERMOT.NS,ASHOKLEY.NS,-2.736406,0.023481,0.028441
4,AUTO,MARUTI.NS,HEROMOTOCO.NS,-2749.871796,0.615714,0.040670
5,BANKING,SBIN.NS,KOTAKBANK.NS,373.610883,-0.022583,0.000364
6,BANKING,ICICIBANK.NS,FEDERALBNK.NS,-39.046388,0.186926,0.000392
7,BANKING,AXISBANK.NS,KOTAKBANK.NS,372.430169,-0.013072,0.000466
8,BANKING,ICICIBANK.NS,KOTAKBANK.NS,367.174930,-0.007578,0.000523
9,BANKING,HDFCBANK.NS,KOTAKBANK.NS,325.521899,0.045525,0.001010


In [ ]:
def backtest_pair_zscore(
    px: pd.DataFrame,
    x: str, y: str,
    alpha: float, beta: float,
    capital: float = 100_000,
    leverage: float = 1.0,          
    entry_z: float = 2.0,
    exit_z: float = 0.5,
    stop_z: float = 4.0,
    roll: int = 60,
    max_hold: int = 40,             
    fee_bps: float = 10.0 # per-leg fee in bps     
):

    df = px[[x, y]].dropna().copy()
    if len(df) < roll + 5:
        return None

    # residual spread using TRAIN alpha/beta on PRICE levels
    resid = df[y] - (alpha + beta * df[x])
    mu = resid.rolling(roll).mean()
    sd = resid.rolling(roll).std(ddof=0)
    z = (resid - mu) / sd

    # trade on yesterday's z
    z_sig = z.shift(1)

    pos = 0  # +1 = long spread (long y, short x*beta), -1 = short spread
    hold_days = 0

    equity = capital
    qty_x = 0.0
    qty_y = 0.0

    fee_rate = (fee_bps) / 10_000.0  # bps -> fraction

    out = []
    prev_px = None

    for t, row in df.iterrows():
        px_x = float(row[x])
        px_y = float(row[y])
        z_t = float(z.loc[t]) if pd.notna(z.loc[t]) else np.nan
        z_s = float(z_sig.loc[t]) if pd.notna(z_sig.loc[t]) else np.nan

        # PnL from holding yesterday's position
        pnl = 0.0
        if prev_px is not None and pos != 0:
            dx = px_x - prev_px[0]
            dy = px_y - prev_px[1]
            # If long spread: +qty_y*dy - qty_x*dx; if short spread: opposite
            pnl = pos * (qty_y * dy - qty_x * dx)

        equity += pnl

        # Determine desired position using signal z_s (yesterday)
        desired = pos

        # stop / max hold
        if pos != 0:
            hold_days += 1
            if np.isfinite(z_s) and abs(z_s) >= stop_z:
                desired = 0
            elif hold_days >= max_hold:
                desired = 0
            elif pos == 1 and np.isfinite(z_s) and z_s >= -exit_z:
                desired = 0
            elif pos == -1 and np.isfinite(z_s) and z_s <= exit_z:
                desired = 0
        else:
            hold_days = 0
            if np.isfinite(z_s) and z_s <= -entry_z:
                desired = 1
            elif np.isfinite(z_s) and z_s >= entry_z:
                desired = -1

        # Execute trades at close (based on yesterday signal)
        trade_cost = 0.0
        if desired != pos:
            # close existing position
            if pos != 0:
                notional_close = abs(qty_y * px_y) + abs(qty_x * px_x)
                trade_cost += fee_rate * notional_close
                qty_x = 0.0
                qty_y = 0.0
            # open new position
            if desired != 0:
                gross = leverage * equity
                leg = gross / 2.0

                qty_y = leg / px_y
                qty_x = (leg * abs(beta)) / px_x  # hedge by beta magnitude

                notional_open = abs(qty_y * px_y) + abs(qty_x * px_x)
                trade_cost += fee_rate * notional_open
                hold_days = 0

            pos = desired
            equity -= trade_cost

        out.append({
            "date": t,
            "px_x": px_x, "px_y": px_y,
            "resid": float(resid.loc[t]),
            "z": z_t,
            "z_signal": z_s,
            "position": int(pos),
            "qty_x": float(qty_x),
            "qty_y": float(qty_y),
            "pnl": float(pnl),
            "trade_cost": float(trade_cost),
            "equity": float(equity)
        })

        prev_px = (px_x, px_y)

    bt = pd.DataFrame(out).set_index("date")
    bt["equity_ret"] = bt["equity"].pct_change().fillna(0.0)
    bt["cummax"] = bt["equity"].cummax()
    bt["drawdown"] = bt["equity"] / bt["cummax"] - 1.0
    return bt

# Backtest
capital_init = 100_000

all_pair_results = []
for _, r in selected_df.iterrows():
    sector, x, y = r["sector"], r["x"], r["y"]
    alpha, beta = float(r["alpha"]), float(r["beta"])

    bt = backtest_pair_zscore(
        test, x, y, alpha, beta,
        capital=capital_init,
        leverage=1.0,
        entry_z=2.0,
        exit_z=0.5,
        stop_z=4.0,
        roll=60,
        max_hold=40,
        fee_bps=10.0
    )
    if bt is None:
        continue

    all_pair_results.append({
        "sector": sector, "x": x, "y": y,
        "alpha": alpha, "beta": beta,
        "bt": bt
    })

print("Backtested pairs:", len(all_pair_results))


Backtested pairs: 35


In [ ]:
def perf_summary(bt: pd.DataFrame):
    eq = bt["equity"]
    ret = bt["equity_ret"]

    total_return = eq.iloc[-1] / eq.iloc[0] - 1.0
    ann_return = ret.mean() * 252
    ann_vol = ret.std(ddof=0) * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan

    max_dd = bt["drawdown"].min()

    # trades
    pos = bt["position"]
    entries = ((pos.shift(1) == 0) & (pos != 0)).sum()
    exits   = ((pos.shift(1) != 0) & (pos == 0)).sum()

    # trade-level PnL (between entry and exit)
    seg_id = (pos != pos.shift(1)).cumsum()
    seg = bt.groupby(seg_id).agg(
        pos=("position","first"),
        pnl=("pnl","sum"),
        cost=("trade_cost","sum"),
        days=("position","size")
    )
    trades = seg[(seg["pos"] != 0)] 
    hit_rate = (trades["pnl"] > 0).mean() if len(trades) else np.nan
    avg_pnl = trades["pnl"].mean() if len(trades) else np.nan

    return {
        "end_equity": float(eq.iloc[-1]),
        "total_return": float(total_return),
        "ann_return": float(ann_return),
        "ann_vol": float(ann_vol),
        "sharpe": float(sharpe),
        "max_dd": float(max_dd),
        "entries": int(entries),
        "exits": int(exits),
        "hit_rate": float(hit_rate) if pd.notna(hit_rate) else np.nan,
        "avg_trade_pnl": float(avg_pnl) if pd.notna(avg_pnl) else np.nan,
        "avg_daily_pnl": float(bt["pnl"].mean()),
        "cost_total": float(bt["trade_cost"].sum()),
    }

metrics = []
for item in all_pair_results:
    bt = item["bt"]
    m = perf_summary(bt)
    m.update({
        "sector": item["sector"],
        "pair": f'{item["y"]} vs {item["x"]}',
        "x": item["x"], "y": item["y"]
    })
    metrics.append(m)

metrics_df = pd.DataFrame(metrics).sort_values(["sharpe","total_return"], ascending=False)
metrics_df


,end_equity,total_return,ann_return,ann_vol,sharpe,max_dd,entries,exits,hit_rate,avg_trade_pnl,avg_daily_pnl,cost_total,sector,pair,x,y
21,111224.050591,0.112241,0.109632,0.055254,1.984167,-0.021913,5,5,1.000000,2135.092429,47.286751,503.063608,IT,WIPRO.NS vs HCLTECH.NS,HCLTECH.NS,WIPRO.NS
31,108208.759633,0.082088,0.081284,0.047139,1.724356,-0.014653,3,4,1.000000,3150.095504,34.933834,454.831313,REALTY,BRIGADE.NS vs GODREJPROP.NS,GODREJPROP.NS,BRIGADE.NS
20,107149.600014,0.071496,0.071952,0.059790,1.203410,-0.025443,4,4,1.000000,2094.774696,30.333777,373.176631,IT,WIPRO.NS vs TCS.NS,TCS.NS,WIPRO.NS
10,107778.654855,0.077787,0.078362,0.066910,1.171156,-0.038418,4,5,1.000000,2091.323711,35.392956,998.798126,ENERGY_UTILS,POWERGRID.NS vs GAIL.NS,GAIL.NS,POWERGRID.NS
12,106421.818890,0.064218,0.064915,0.057700,1.125054,-0.032760,6,7,0.833333,1203.351022,30.052304,1031.152499,ENERGY_UTILS,IOC.NS vs ONGC.NS,ONGC.NS,IOC.NS
9,105897.422758,0.058974,0.059769,0.055556,1.075842,-0.050692,3,4,1.000000,1784.629017,25.055822,316.421022,BANKING,KOTAKBANK.NS vs HDFCBANK.NS,HDFCBANK.NS,KOTAKBANK.NS
5,105714.676441,0.057147,0.058040,0.056030,1.035874,-0.052079,3,4,1.000000,1739.880976,24.289735,309.177888,BANKING,KOTAKBANK.NS vs SBIN.NS,SBIN.NS,KOTAKBANK.NS
7,105725.186673,0.057252,0.058151,0.056213,1.034490,-0.052173,3,4,1.000000,1724.693415,24.320586,306.318565,BANKING,KOTAKBANK.NS vs AXISBANK.NS,AXISBANK.NS,KOTAKBANK.NS
8,105732.018990,0.057320,0.058233,0.056490,1.030842,-0.052791,3,4,1.000000,1719.564982,24.341113,304.577041,BANKING,KOTAKBANK.NS vs ICICIBANK.NS,ICICIBANK.NS,KOTAKBANK.NS
29,109573.975762,0.095740,0.100582,0.124633,0.807023,-0.057448,3,3,1.000000,4002.787146,43.857624,1302.715068,PHARMA,APOLLOHOSP.NS vs CIPLA.NS,CIPLA.NS,APOLLOHOSP.NS
